Extraindo PDF

In [1]:
!pip install transformers datasets pdfplumber accelerate sentencepiece


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
import pdfplumber
from pathlib import Path
# from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

c:\Users\rayco\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:

def extract_text_from_pdf(pdf_path, skip_pages=6):
    """Extrai texto de um arquivo PDF."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            if i < skip_pages: # Ignora as primeiras páginas (capa, sumário, etc.)
                continue
            page_text = page.extract_text()
            if page_text:
                pages.append(page_text.strip())
    return "\n\n".join(pages)

# Substitua pelo caminho do seu PDF
pdf_path = "20260018701.pdf"
full_text = extract_text_from_pdf(pdf_path)
print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Total de caracteres extraídos: 243128

--- INÍCIO DO TEXTO ---

Amedicinadasimplicidade|7
A medicina da simplicidade
– trabalhar com plantas é a
ciência do simples
Alésio dos Passos Santos – ambientalista, cultivador e edu-
cadordeplantasmedicinais(comcontribuiçõesdeCésarPaulo
Simionatto,MuriloLeandroMarcos,LeilaNerydosSantosSouza,
Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon
Bittencourt)
Amedicinadosinergismoeaenergiadecura
Osinergismoéumadascoisasmaisimportantesnomundo
dasplantas.Oremédiodefarmácia,oquequeé?Umprincípio
ativoisolado,concentrad


Dividindo em chunks

In [19]:
import re

def limpar_texto(texto):
    """
    Remove espaços extras e normaliza o texto bruto extraído do PDF.
    Isso ajuda a eliminar ruídos comuns em extrações de arquivos PDF.
    """
    # Substitui múltiplas quebras de linha ou espaços por um único espaço e remove espaços no início e no fim
    return re.sub(r'\s+', ' ', texto).strip()

def dividir_em_paragrafos(texto):
    """
    Divide o texto em parágrafos significativos.
    """
    # Dividindo os paragrafos por quebra de linha e limpando espaços extras
    paragrafos = texto.split("\n")
    # Filtro de ruído: ignora fragmentos menores que 50 caracteres (como números de página) 
    return [p.strip() for p in paragrafos if len(p.strip()) > 50]


In [20]:
def chunk_text_especializado(paragrafos, max_chunk_chars=500, overlap_chars=100):
    """
    Agrupa parágrafos em blocos menores (chunks) com sobreposição.
    
    Sugestões aplicadas:
    - max_chunk_chars=500: Chunks menores tendem a ser mais focados.
    - overlap_chars: Evita a perda de contexto nas 'emendas' entre blocos.
    """
    chunks = []
    chunk_atual = ""

    for paragrafo in paragrafos:
        # TRATAMENTO DE INTEGRIDADE: 
        # Se um parágrafo sozinho for maior que o limite máximo, ele é dividido em partes.
        if len(paragrafo) > max_chunk_chars:
            if chunk_atual:
                chunks.append(chunk_atual)
                chunk_atual = ""
            # Divide o parágrafo gigante respeitando o overlap
            for i in range(0, len(paragrafo), max_chunk_chars - overlap_chars):
                chunks.append(paragrafo[i : i + max_chunk_chars])
            continue

        # Lógica de agrupamento com verificação de limite
        if len(chunk_atual) + len(paragrafo) + 1 <= max_chunk_chars:
            chunk_atual += (" " if chunk_atual else "") + paragrafo 
        else:
            if chunk_atual:
                chunks.append(chunk_atual)
            
            # APLICAÇÃO DE SOBREPOSIÇÃO (Sliding Window):
            # O novo chunk começa com o final do chunk anterior para manter o contexto [2].
            inicio_com_contexto = chunk_atual[-overlap_chars:] if len(chunk_atual) > overlap_chars else ""
            chunk_atual = (inicio_com_contexto + " " if inicio_com_contexto else "") + paragrafo

    if chunk_atual:
        chunks.append(chunk_atual)

    return chunks


In [21]:
paragrafos = dividir_em_paragrafos(full_text)
print(f"\nParágrafos encontrados: {len(paragrafos)}")


Parágrafos encontrados: 3133


In [22]:
chunks = chunk_text_especializado(paragrafos, max_chunk_chars=600)
print(f"Número de chunks: {len(chunks)}")
print(f"\nExemplo de chunk (primeiro):\n{chunks[0][:600]}...")

Número de chunks: 382

Exemplo de chunk (primeiro):
pessoas. Sua publicação foi viabilizada pelo Projeto “Capaci- taçãodeProfissionaisdaAtençãoBásicadeFlorianópolis”,do Ministério da Saúde de 2013. Contribuíram para seu nasci- mento profissionais e gestores da rede municipal de saúde (SUSFlorianópolis);professores,técnicoseestudantesdaUni- versidade Federal de Santa Catarina (UFSC); apoiadores do HortoDidáticodePlantasMedicinaisdoHospitalUniversitário da UFSC e sobretudo moradores das comunidades de Flori- anópolis,quehámuitosséculosmantémvivaatradiçãoances- tral da cura pelas plantas – compartilhando sua valiosa...


Carregando modelo

In [23]:
# --- Modelo Causal ---
causal_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
causal_tokenizer = AutoTokenizer.from_pretrained(causal_id)
causal_tokenizer.pad_token = causal_tokenizer.eos_token  # GPT-2 não tem pad_token
causal_model = AutoModelForCausalLM.from_pretrained(causal_id)

generator_causal = pipeline(
    "text-generation",
    model=causal_model,
    tokenizer=causal_tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.3
)

print("Modelo carregado com sucesso.")

c:\Users\rayco\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rayco\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 820.92it/s]
[transforme

Modelos carregados com sucesso.


In [29]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def is_bad_chunk(text, min_len=80):
    text = text.strip()
    if len(text) < min_len:
        return True
    words = text.split()
    if len(set(words)) < 10:
        return True
    strange_ratio = sum(1 for c in text if not c.isalnum() and c not in " .,;:-()[]{}") / max(len(text), 1) #
    if strange_ratio > 0.3:
        return True
    return False

def run_model(generator, prompt):
    """Executa qualquer pipeline e retorna o texto gerado."""
    output = generator(prompt, max_new_tokens=256, temperature=0.3, do_sample=True)
    if isinstance(output, list):
        item = output[0]
        if isinstance(item, dict):
            return item.get('generated_text') or item.get('summary_text') or str(item)
        return str(item)
    return str(output)

def extract_triple(generated_text, chunk):
    """Tenta extrair a tripla da saída do modelo, com fallback."""
    # 1. Tenta JSON
    json_match = re.search(r'\{[\s\S]*\}', generated_text)
    if json_match:
        try:
            data = json.loads(json_match.group(0))
            instruction = str(data.get("instruction", "")).strip()
            input_text = str(data.get("input", "")).strip()
            output = str(data.get("output", "")).strip()
            if instruction and output:
                return {"instruction": instruction, "input": input_text, "output": output}
        except:
            pass
    # 2. Regex por campos
    instruction = input_text = output = ""
    for key in ["instruction", "input", "output"]:
        pattern = rf'{key}\s*[:=-]\s*(.*)'
        match = re.search(pattern, generated_text, re.IGNORECASE)
        if match:
            if key == "instruction": instruction = match.group(1).strip()
            elif key == "input": input_text = match.group(1).strip()
            elif key == "output": output = match.group(1).strip()
    # 3. Fallback
    if not instruction:
        instruction = "Explique o conteúdo do texto."
    if not input_text:
        input_text = chunk[:500]
    if not output:
        output = re.sub(r'\s+', ' ', generated_text)[:200]
    return {"instruction": instruction, "input": input_text, "output": output}

In [30]:
clean_chunks = [clean_text(c) for c in chunks if not is_bad_chunk(c)]
print(f"Chunks válidos: {len(clean_chunks)}")

Chunks válidos: 382


In [36]:
PROMPT_CAUSAL = """<|system|>
Você é um especialista em plantas medicinais. Sua tarefa é ler um trecho de texto e gerar uma pergunta e resposta sobre ele.
</s>
<|user|>
Leia o trecho abaixo e gere um JSON com dois campos:
- "Instruction": uma pergunta objetiva sobre o conteúdo do trecho
- "Output": a resposta para essa pergunta, baseada apenas no trecho

Exemplo de saída esperada:
{
  "Instruction": "Para que é indicado o uso da camomila?",
  "Output": "A camomila é indicada para acalmar o sistema nervoso, aliviar cólicas e auxiliar no sono."
}

Agora gere para o trecho abaixo:

Trecho:
{chunk}

Responda APENAS com o JSON, sem mais nada.
</s>
<|assistant|>
{{"""

In [32]:
triplas_causal = []
amostra_chunks = clean_chunks[:50]  # para demonstração, processamos apenas 50 chunks

for i, chunk in enumerate(amostra_chunks):
    prompt = PROMPT_CAUSAL.replace("{chunk}", chunk)
    try:
        result = run_model(generator_causal, prompt)
        # O modelo causal pode repetir o prompt; removemos a parte inicial se existir
        if result.startswith(prompt):
            generated = result[len(prompt):].strip()
        else:
            generated = result
        triple = extract_triple(generated, chunk)
        if len(triple["output"]) > 20 and len(triple["instruction"]) > 5:
            triplas_causal.append(triple)
        if i % 10 == 0:
            print(f"Processados {i}/{len(amostra_chunks)} chunks (causal)")
    except Exception as e:
        print(f"Erro no chunk {i}: {e}")

print(f"Triplas geradas (causal): {len(triplas_causal)}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processados 0/50 chunks (causal)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Processados 10/50 chunks (causal)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

...

Processados 30/50 chunks (causal)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Processados 40/50 chunks (causal)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Triplas geradas (causal): 44


In [33]:
def clean_triple(triple):
    for key in triple:
        triple[key] = triple[key].strip()
    if triple["input"].lower() in ("none", "null", "n/a", ""):
        triple["input"] = ""
    if len(triple["output"]) < 20 or len(triple["instruction"]) < 5:
        return None
    return triple

def limpar_lista(triplas):
    return [t for t in (clean_triple(x) for x in triplas) if t is not None] # Filtra triplas inválidas após limpeza


triplas_causal = limpar_lista(triplas_causal)

print(f"Triplas causal após limpeza: {len(triplas_causal)}")

Triplas causal após limpeza: 44


In [34]:
def salvar_jsonl(triplas, nome_arquivo):
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        for t in triplas:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"Dataset salvo em {nome_arquivo}")

salvar_jsonl(triplas_causal, "dataset_causal_500chars.jsonl")

Dataset salvo em dataset_causal_500chars.jsonl


In [35]:
def mostrar_exemplo(triplas, titulo, n=3):
    print(f"\n=== {titulo} ===")
    for i, t in enumerate(triplas[:n], 1):
        print(f"\n--- Exemplo {i} ---")
        print(json.dumps(t, indent=2, ensure_ascii=False))

mostrar_exemplo(triplas_causal, "TinyLlama/TinyLlama-1.1B-Chat-v1.0")


=== TinyLlama/TinyLlama-1.1B-Chat-v1.0 ===

--- Exemplo 1 ---
{
  "instruction": "pergunta que um usuário faria sobre o texto",
  "input": "",
  "output": "exemplo de resposta para a pergunta"
}

--- Exemplo 2 ---
{
  "instruction": "Pergunta que um usuário faria sobre o texto",
  "input": "",
  "output": "O texto contém informações sobre a cura das plantas, o que significa que a inter-relação pode ser comuns entre a área de ciências biológicas e a área de ciências da saúde."
}

--- Exemplo 3 ---
{
  "instruction": "Explique o conteúdo do texto.",
  "input": "eberBiavatti:ProfessoraDoutoradoDepartamento Mayara Krasinski Caddah: Professora Doutora do Departa- MelissaCostaSantos:FarmacêuticadaPrefeituraMunicipalde FlorianópolisedaComissãodePráticasIntegrativaseComple- MuriloLeandroMarcos:Médicodefamíliaecomunidadedocen- PráticasIntegrativaseComplementaresdeFlorianópolis. ShirleyRosa:FarmacêuticacolaboradoradoHortodePlantas Prainha - Cláudia Schenya, Fernanda Nowak, Gisele Damian Antonio